# **ДЗ 1 из 2. Линейные модели: веса, регуляризация и что они прогнозируют**

### Что внутри

Ноутбук — практика к урокам **«Линейная регрессия»** и **«Что прогнозирует регрессия?»**. Пройдём по всей цепочке вопросов о весах линейной модели:

1. как читать веса как вклад признака (барплот, относительная важность);
2. почему одной точечной оценки веса мало — важность через кросс-валидацию, `MEAN/SE` и устойчивая к выбросам `MEDIAN`-версия;
3. как диагностировать модель по остаткам;
4. мультиколлинеарность и VIF — куда девается вес, если признаки коррелируют;
5. что вообще прогнозирует модель в зависимости от loss-функции (MSE → среднее, quantile loss → квантиль);
6. смысл свободного члена $w_0$ и что делает с весами регуляризация (Ridge, Lasso, L0).

$$y = \sum_{i=1}^{n}x_i\beta_i + \beta_0 = \beta_0 + x_1\beta_1+....+x_n\beta_n + \varepsilon$$

### **Напоминание: ограничения применения модели**

В теории мы выделили 4+4 ограничения. Первые 4 — неформально, «смысловые»:
___
1. Признаки должны быть осмысленными — то есть есть понимание, что умножается на вес.
2. Масштаб признаков сопоставим между собой.
3. Коэффициенты модели устойчивы — нужно не всегда, но на базовых задачах — желательное свойство.
4. Зависимости между признаками не должны разрушать модель — при сильной корреляции признаков коэффициенты линейной модели могут становиться нестабильными.
___
Вторые четыре задают контекст, позволяющий говорить, верна ли оценка весов математически:

1. **Линейность** — условное среднее целевой переменной $y$ линейно связано c признаками $x_i$: $$\mathbb{E}[Y|X] = \beta_0 + \sum_i x_i\beta_i$$

   При нелинейности для исправления ситуации можно пробовать преобразования данных. Напомним, что стадартные:

      - **Для непрерывных признаков:** логарифмирование, извлечение корня, `z-score`-преобразование, `StandardScaler`, `MinMaxScaler` и [другие](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.preprocessing).

      - **Для категориальных признаков:** One-Hot encoding. Для избежания линейной зависимости обычно удаляют столбец одной из категорий — но только если в датасете представлены ВСЕ возможные категории (иначе останутся строки, где все OHE-столбцы равны 0).

      - Также подходы можно комбинировать — например, разбить непрерывный признак на квантили ([`pd.cut`, `pd.qcut`](https://pbpython.com/pandas-qcut-cut.html)).

2. **Независимость объектов выборки** — объекты должны попадать в выборку независимо друг от друга.
3. **Отсутствие мультиколлинеарности признаков** — корреляция признаков друг с другом слабая или отсутствует.
4. **Нормальность остатков и гомоскедастичность ошибок** — два разных, но важных предположения о неспрогнозированной части модели. Нормальность означает, что ошибки $\varepsilon = y - \hat{y}$ примерно нормально распределены. Гомоскедастичность означает, что дисперсия этих ошибок примерно одинакова при разных значениях признаков или предсказаний: $Var(\varepsilon|X)=\sigma^2$.

В домашнем задании мы посмотрим, как разные артефакты реального мира (корреляция, неравноменость данных) влияют на устойчивость и веса регресии, а также руками почувствуем все блоки теории, которые изучили. Вперед!

Для начала — просто обучим модель.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.datasets import fetch_california_housing

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, r2_score

Фиксируем случайность.

In [ ]:
RANDOM_STATE = 42

В качестве набора данных используем встроенный в sklearn [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset). Целевая переменная `y` — медианная стоимость дома в сотнях тысяч долларов (США).

In [ ]:
data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target

X.head()

Разделим данные на тренировочную и тестовую выборки, не забыв про масштабирование. 

**Q1: Зачем масштабирование применяется для линейных моделей?** 

`Выберите все верные утверждения в тренажере`

**Q2: Как ведёт себя `StandardScaler()` масштабирование?**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.25)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Обучим линейную регрессию и сравним предсказание с базовым — всем домам прогнозируем среднее по обучающей выборке.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

predictions = lr.predict(X_test_scaled)
base = np.array([y_train.mean()]*len(X_test)) # прогнозируем среднее

print('Качество базового алгоритма (MAE): ', mean_absolute_error(y_test, base))
print('Качество линейной регрессии (MAE): ', mean_absolute_error(y_test, predictions))

In [ ]:
print('Качество базового алгоритма (R2): ', r2_score(y_test, base))
print('Качество линейной регрессии (R2): ', r2_score(y_test, predictions))

Не отлично, но и не ужасно! Регрессия построена, она лучше базовой, но всё же не идеальна — это связано с природой данных. 

## Блок 2. Веса как вклад признака

После обучения линейной регрессии у нас есть веса — сила вклада каждого признака в прогноз. Они хранятся в атрибуте `lr.coef_`.

In [ ]:
labels = X_train.columns # названия признаков
values = lr.coef_ # значения весов признаков

Посмотрим на табличные значения.

In [ ]:
weights_data = pd.DataFrame(values, index=labels, columns=['weight'])

weights_data

**Q3.** Какой признак является самым значимым в модели по абсолютному значению веса?

Веса могут быть как положительными, так и отрицательными — знак отражает направление связи. Способ представить их визуально — барплот. Первый вариант, сохраняя знак:

In [ ]:
plt.figure(figsize=(12, 4))

bar = plt.bar(height=lr.coef_, x=labels)
plt.bar_label(bar, padding=-13, color='black')

plt.title('Важность признаков линейной регрессии на основе весов');

Второй — абсолютные значения:

In [ ]:
plt.figure(figsize=(12, 4))

bar = plt.bar(height=np.abs(values), x=labels)
plt.bar_label(bar, color='black')

plt.title('Важность признаков линейной регрессии на основе весов (модуль)');

## Блок 2.1. Относительная важность признаков

Более удобное представление вклада признака можно посчитать «в лоб» — нормировав каждый коэффициент на сумму абсолютных значений всех весов. Получаем долю каждого признака в суммарном «весе» модели — удобно, когда нужно донести относительную значимость.

In [ ]:
weights_data2 = pd.DataFrame([labels, values]).T
weights_data2.columns = ['feature', 'feature_weight']

weights_sum2 = sum(abs(weights_data2['feature_weight']))

weights_data2['feature_weight_normalized'] = weights_data2['feature_weight'].apply(lambda x: round(x/weights_sum2*100, 2))

In [ ]:
weights_data2.sort_values(by='feature_weight_normalized', ascending=False)

**Q4.** Проанализируйте относительные вклады признаков в модель. Выберите верные утверждения

## Блок 3. Важность через кросс-валидацию: `MEAN(β)/SE(β)`

В теории урока мы считали важность признака как $|MEAN(\beta)/SE(\beta)|$ по нескольким итерациям обучения (пример с кофейнями). Веса из Блока 2 — это ровно **одна** такая итерация, на одном конкретном разбиении `train/test`. Насколько она надёжна? Проверим, повторив обучение на разных фолдах — так же, как это будет сделано во втором ДЗ для логистической регрессии, только здесь мы делаем это первыми.

Обучим модель на 9 разных подвыборках (3 объекта `KFold` с разным `random_state`, по 3 фолда каждый) и соберём веса.

In [ ]:
coefs = []

for rs in [12, 7, 13]:
    kf = KFold(n_splits=3, shuffle=True, random_state=rs)
    for train_idx, val_idx in kf.split(X_train_scaled):
        model = LinearRegression()
        model.fit(X_train_scaled[train_idx], y_train.values[train_idx])
        coefs.append(model.coef_)

coefs = np.array(coefs) # (9, n_features)
coefs.shape

Посчитаем `MEAN(β)`, `SE(β)` (обычное стандартное отклонение по итерациям — как в примере с кофейнями в теории) и важность $|\frac{MEAN}{SE}|$.

In [ ]:
mean_ = coefs.mean(axis=0)
se_ = coefs.std(axis=0, ddof=1)
importance_mean = np.abs(mean_ / se_)

cv_report = pd.DataFrame({
    'feature': labels,
    'point_estimate': values,
    'MEAN(beta)_cv': mean_,
    'ABS(MEAN(beta)_cv)': np.abs(mean_),
    'SE(beta)_cv': se_,
    'importance_mean=|MEAN/SE|': importance_mean,
})

cv_report.sort_values('importance_mean=|MEAN/SE|', ascending=False)

**Q5.** Какой признак — самый важный по оценке $|MEAN/SE|$ Совпадает ли это с самым большим по модулю весом из Блока 2 (`Latitude`)?

**Q6.** Какая пара признаков обладает самым большим **абсолютным И, средним И se**?

In [ ]:
cv_report.sort_values(['ABS(MEAN(beta)_cv)', 'SE(beta)_cv'], ascending=False)

### Устойчивая версия: медиана вместо среднего

В уроке «Что прогнозирует регрессия?» мы выяснили: среднее чувствительно к выбросам, медиана — устойчива к ним (MSE-оптимум — среднее, MAE-оптимум — медиана). Тот же принцип работает и здесь: если хотя бы одна из 9 итераций CV случайно попала на «шумный» фолд, `MEAN` и `SE` это увидят и сдвинутся, а медиана — нет.

Посчитаем `MEDIAN(β)` и робастный аналог `SE` — **MAD** (median absolute deviation), масштабированный на $1.4826$ (для нормального распределения это делает MAD сопоставимым по масштабу со стандартным отклонением).

In [ ]:
median_ = np.median(coefs, axis=0)
mad_ = np.median(np.abs(coefs - median_), axis=0) * 1.4826
importance_median = np.abs(median_ / mad_)

cv_report['MEDIAN(beta)_cv'] = median_
cv_report['robust_SE(MAD*1.4826)'] = mad_
cv_report['importance_median=|MEDIAN/robustSE|'] = importance_median

cv_report.sort_values('importance_median=|MEDIAN/robustSE|', ascending=False) #.head(5)

In [ ]:
cv_report.sort_values('importance_mean=|MEAN/SE|', ascending=False)

**Q7.** Сравните два ранжирования — по $|MEAN/SE|$ и по $|MEDIAN/\text{robust SE}|$. Совпадают ли первые 5 признаков? 

* **Обратите внимание на изменение веса важности AveOccup. Достаньте коэффициенты признака.**

In [ ]:
coefs[:, 5] # AveOccup weighs

## Блок 4. Диагностика по остаткам

В теории урока сказано: после обучения нужно проверить остатки на нормальность и гомоскедастичность. Посмотрим на это на практике — сначала прогноз против факта.

In [ ]:
plt.scatter(predictions, y_test)
plt.xlabel('Прогноз')
plt.ylabel('Факт')
plt.title('Диаграмма рассеивания предсказанных значений и реальных данных');

На этом шаге вы уже, скорее всего, видите причину, но мы детализируем. Разобьём объекты по величине остатка на группы (`low` / `middle` / `high` — по квартилям абсолютного остатка) и посмотрим, где они концентрируются относительно одного из признаков.

In [ ]:
residuals = pd.Series(abs(predictions - y_test))

q1 = residuals.quantile(0.25)
q2 = residuals.quantile(0.75)

def get_resid_class(x, q1=q1, q2=q2):
    if x <= q1:
        return 'low'
    elif x <= q2:
        return 'middle'
    else:
        return 'high'

residuals_class = residuals.apply(get_resid_class)

In [ ]:
sns.scatterplot(x=X_test['MedInc'], y=y_test, hue=residuals_class);
plt.legend();
plt.title('Residual class based on the MedInc feature and the target variable.', 
          pad=15);

**Q8.** Проанализируйте график. Какое (или какие) противоречие мешает линейности связи, и где оно наблюдается? Выберите верные утверждения.

## Блок 5. Мультиколлинеарность

В Блоке 3 мы увидели: `Latitude` — лидер по точечной оценке, но не по устойчивости к пересэмплированию и из теории уроков мы знаем, что такое поведение может быть следствием мультиколлинеарности. Проверим, так ли это. 

**Q9\*.** Посчитайте корреляцию между `Latitude` и `Longitude` на `X_train`, а затем — VIF (variance inflation factor) для всех признаков.

В поле ответа введите значение корреляции, округленное до сотых. 

**Важно:** `variance_inflation_factor` из `statsmodels` требует явно добавленной константы (`sm.add_constant`) — без неё числа будут сильно завышены и бессмысленны.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

print('Корреляция Latitude/Longitude:', X_train['Latitude'].corr(X_train['Longitude']))

X_train_c = sm.add_constant(X_train)
vif_data = pd.DataFrame()
vif_data['feature'] = X_train_c.columns
vif_data['VIF'] = [variance_inflation_factor(X_train_c.values, i) for i in range(X_train_c.shape[1])]
vif_data.round(2)

**Q10**. VIF и $R^2$ 

VIF и $R^2$ вспомогательной регрессии — это одно и то же число в двух шкалах:

$$VIF_j = \frac{1}{1-R_j^2}, \qquad R_j^2 = 1 - \frac{1}{VIF_j},$$

где $R_j^2$ — качество регрессии признака $x_j$ на все остальные признаки (именно эта регрессия и считается внутри variance_inflation_factor). Восстановите $R_j^2$ по уже посчитанным VIF (только по формуле), а затем проверьте себя: обучите LinearRegression для Latitude и Longitude от остальных признаков напрямую и сравните .score() с тем, что получили по формуле.

В ответ укажите сумму $R^2_{Latitude} + R^2_{Longitude}$, округленную до сотых. 

In [ ]:

# Восстановление R^2 по формуле — без единой новой модели
vif_data['R2_from_VIF'] = 1 - 1 / vif_data['VIF']
vif_data.round(4)

# Проверка прямой регрессией: Latitude ~ остальные признаки
X_other = X_train.drop(columns=['Latitude'])
y_target = X_train['Latitude']

aux_model = LinearRegression().fit(X_other, y_target)
r2_direct = aux_model.score(X_other, y_target)

vif_latitude = vif_data.loc[vif_data['feature'] == 'Latitude', 'VIF'].values[0]
r2_from_vif = 1 - 1/vif_latitude

print('R^2 напрямую (LinearRegression.score)     :', round(r2_direct, 4))
print('R^2 восстановлен по формуле 1 - 1/VIF      :', round(r2_from_vif, 4))

In [ ]:
0.8907 + 0.8866

In [ ]:
# Проверка прямой регрессией: Latitude ~ остальные признаки
X_other = X_train.drop(columns=['Longitude'])
y_target = X_train['Longitude']

aux_model = LinearRegression().fit(X_other, y_target)
r2_direct = aux_model.score(X_other, y_target)

vif_longtitide = vif_data.loc[vif_data['feature'] == 'Longitude', 'VIF'].values[0]
r2_from_vif = 1 - 1/vif_longtitide

print('R^2 напрямую (LinearRegression.score)     :', round(r2_direct, 4))
print('R^2 восстановлен по формуле 1 - 1/VIF      :', round(r2_from_vif, 4))

## Блок 6. Что прогнозирует регрессия? Проверяем на практике

В теории урока разобрано: MSE-модель учится прогнозировать **условное среднее** $\mathbb{E}[Y\mid X=x]$, а модель с pinball (quantile) loss для $\tau$ — **$\tau$-квантиль** условного распределения. При $\tau=0.5$ это медиана — тот же функционал, который элицитирует MAE.

Наш `target` (медианная стоимость дома) скошен вправо (skew $\approx 0.98$: mean $\approx 2.07$, median $\approx 1.80$) — то есть среднее и медиана заметно различаются. Значит, если теория верна, модели с разными loss-функциями должны в среднем прогнозировать по-разному.

**Q11.** Обучите на `X_train_scaled`, `y_train`:
1. `LinearRegression` (эталон MSE-оптимума → должна целиться в среднее).
2. `sklearn.linear_model.QuantileRegressor(quantile=0.5, alpha=0, solver='highs')` (медиана).
3. `QuantileRegressor(quantile=0.1, ...)` и `QuantileRegressor(quantile=0.9, ...)`.

Сравните средние прогнозы каждой модели на `X_test_scaled` с `y_test.mean()` и `y_test.median()`.
В ответе выберите модель, которая прогнозирует наибольшее значение среднего. 

In [ ]:
from sklearn.linear_model import QuantileRegressor

pred_mse = lr.predict(X_test_scaled)

qr50 = QuantileRegressor(quantile=0.5, alpha=0, solver='highs').fit(X_train_scaled, y_train)
qr10 = QuantileRegressor(quantile=0.1, alpha=0, solver='highs').fit(X_train_scaled, y_train)
qr90 = QuantileRegressor(quantile=0.9, alpha=0, solver='highs').fit(X_train_scaled, y_train)

pred_median = qr50.predict(X_test_scaled)
pred_q10 = qr10.predict(X_test_scaled)
pred_q90 = qr90.predict(X_test_scaled)

mae_mse_model = np.mean(np.abs(pred_mse - y_test.values))
mae_median_model = np.mean(np.abs(pred_median - y_test.values))
print('MAE MSE-модели:      ', round(mae_mse_model, 4))
print('MAE медианной модели:', round(mae_median_model, 4))

In [ ]:
print('y_train.mean():  ', round(y_train.mean(), 3),
    ' | y_train.median():', round(y_train.median(), 3), 
    ' | y_train.quantile(0.5) :', round(y_train.quantile(0.5), 3),
    ' | y_train.quantile(0.1) :', round(y_train.quantile(0.1), 3),
    ' | y_train.quantile(0.9) :', round(y_train.quantile(0.9), 3),
    )

print('y_test.mean():  ', round(y_test.mean(), 3),
    ' | y_test.median():', round(y_test.median(), 3), 
    ' | y_test.quantile(0.5) :', round(y_test.quantile(0.5), 3),
    ' | y_test.quantile(0.1) :', round(y_test.quantile(0.1), 3),
    ' | y_test.quantile(0.9) :', round(y_test.quantile(0.9), 3),
    )


print('MSE-модель, средний прогноз:      ', round(pred_mse.mean(), 3))

print('Quantile tau=0.5, средний прогноз:', round(pred_median.mean(), 3))
print('Quantile tau=0.1, средний прогноз:', round(pred_q10.mean(), 3))
print('Quantile tau=0.9, средний прогноз:', round(pred_q90.mean(), 3))


## Блок 7. Смысл $\beta_0$ и что делает регуляризация

В теории урока выведено: $\beta_0$ — это **не** матожидание целевой переменной «вообще», а прогноз модели, когда все признаки равны своему среднему (или нулю — если была стандартизация). 

Мы стандартизировали признаки в самом начале — значит, для нашей модели, при условии хорошего обучения (с маленькими ошибками (остатками)) должно выполняться:

$$\beta_0 = lr.intercept\_ \approx \bar y_{train}.$$

Проверим.

In [ ]:
print('OLS (MSE): lr.intercept_ =', round(lr.intercept_, 4), 
      ' | y_train.mean() =', round(y_train.mean(), 4),
      ' | diff =', round(lr.intercept_ - y_train.mean(), 6))
print()

for tau, model, name in [(0.1, qr10, 'qr10'), (0.5, qr50, 'qr50'), (0.9, qr90, 'qr90')]:
    q_train = y_train.quantile(tau)
    print(f'{name} (tau={tau}): intercept_ = {model.intercept_:.4f} | y_train.quantile({tau}) = {q_train:.4f} | diff = {model.intercept_ - q_train:.4f}')

Для OLS равенство `lr.intercept_` $\approx$ `y_train.mean()` держится с достаточной точностью. А вот для других видов регресии мы видим значимые отличия. Давайте проверим, как ошибаются наши модели. 

In [ ]:
# У OLS первоусловие — "сумма остатков на train равна нулю" (значение).
# У quantile-регрессии первоусловие другое — "доля train-остатков ниже нуля равна tau" (ранг, не значение). # TO DO: понять это

for tau, model, name in [('none', lr, 'OLS'), (0.1, qr10, 'qr10'), (0.5, qr50, 'qr50'), (0.9, qr90, 'qr90')]:
    resid_train = y_train.values - model.predict(X_train_scaled)
    frac_below = (resid_train < 0).mean()
    total_sum = (resid_train).sum()
    print(f'{name}: доля train-остатков < 0 = {frac_below:.4f}  (ожидаем tau={tau}), total_sum={total_sum:.4f}')

**Что случилось?**

1. OLS. 
    - total_sum=0.0000 — это тождество, которое даёт $\beta_0=\bar y_{train}$ (уравнение МНК). 
    - доля остатков <0 — 0.5858: у 58.6% домов модель завысила цену, и только у 41.4% занизила. OLS это не волнует — условие оптимальности этой функции — минимизация квадрата ошибок. Мы видели, что таргет скошен вправо (Блок 6), отсюда MSE-модели выгоднее слегка завышать цену для основной массы недорогих домов, чтобы не платить квадратичный штраф за недооценку немногих дорогих.


2. Quantile-модели. 
    - Доля остатков <0 близка к  $\tau$  (0.0996, 0.4997, 0.8998) — это их настоящая гарантия, доказанная в теории урока. Мы сказали, что модели выгодно прогнозировать $\tau$-квантиль. Тогда по определению, если $\hat y$ — это $\tau$%-квантиль ($\tau=0.1$), то по определению только $\tau$% значений $Y$ лежат ниже этого порога, а 90% — выше. То есть:

    $$P(Y < \hat y) = \tau, \qquad P(Y > \hat y) = 1-\tau$$
    
    Остаток — это resid = y - pred. Значит:
    - resid < 0 $\iff$ y < pred $\iff$ объект ниже предсказанного квантиля
    - resid > 0 $\iff$ y > pred $\iff$ объект выше предсказанного квантиля


    Отсюда $P(resid<0) = P(y<pred) = \tau$

    - высокие значения total_sum следствие того, что pinball loss не штрафует за размер ошибки.


Обобщая: каждая loss-функция на train гарантирует ровно тот вид баланса остатков и ту интерпретацию, которая следует из её оптимального решения. 

**Бонус.** Что произойдёт с равенством OLS, если обучить модель на **не**стандартизированных признаках (сырых `X_train`, без `StandardScaler`)? Останется ли $w_0 \approx \bar y_{train}$? Проверьте на практике и объясните результат, опираясь на формулу $w_0 = \bar y - \sum_i w_i \bar x_i$ из теории.

Это задание не проверяется и его можно пропустить =) 

In [ ]:
lr_raw = LinearRegression()
lr_raw.fit(X_train, y_train)  # НЕ стандартизированные признаки

print('lr_raw.intercept_        :', lr_raw.intercept_)
print('y_train.mean()           :', y_train.mean())
print('diff                     :', lr_raw.intercept_ - y_train.mean())
print()

# Общая формула w0 = ybar - sum(w_i * xbar_i) должна выполняться всегда, вне зависимости от масштабирования
manual_w0 = y_train.mean() - np.sum(lr_raw.coef_ * X_train.mean().values)
print('w0 по формуле ybar - sum(w_i * xbar_i):', manual_w0)
print('lr_raw.intercept_ (сверка)            :', lr_raw.intercept_)
print('diff                                   :', manual_w0 - lr_raw.intercept_)

### Ridge: явное решение

Мы только что убедились: `Latitude` и `Longitude` сильно коррелируют, а их VIF заметно выше порога — одна из причин нестабильности весов (Блок 3, 5). Ridge-регрессия имеет явное решение:

$$\hat\beta^{Ridge}_\lambda = (X^TX+\lambda I)^{-1}X^Ty.$$

Откуда:

$$(X^TX+\lambda I)\hat\beta^{Ridge}_\lambda = X^Ty$$

**Q12.** На тех же `X_train_scaled`, `y_train`:
1. Реализуйте формулу руками (`numpy`, без `sklearn`) для $\lambda=1.0$.
2. Сравните с `sklearn.linear_model.Ridge(alpha=1.0, fit_intercept=False)`, обученной на тех же данных.
3. Сравните веса `Latitude`/`Longitude` в обычной `LinearRegression` (`lr.coef_`) и в Ridge — сильно ли они изменились по сравнению с остальными признаками?

Чтобы найти решение системы, вам потребуется `np.linalg.solve(A, b)`, находящий решение системы $Ax = b.$

In [ ]:
from sklearn.linear_model import Ridge

lam = 1.0
n_features = X_train_scaled.shape[1]

# Явное решение
beta_ridge_manual = np.linalg.solve(
    X_train_scaled.T @ X_train_scaled + lam * np.eye(n_features),
    X_train_scaled.T @ y_train
)

# sklearn
ridge_sklearn = Ridge(alpha=lam, fit_intercept=False)
ridge_sklearn.fit(X_train_scaled, y_train)

comparison = pd.DataFrame({
    'feature': X.columns,
    'beta_OLS (lr.coef_)': lr.coef_.round(4),
    'beta_ridge_manual': beta_ridge_manual.round(4),
    'beta_ridge_sklearn': ridge_sklearn.coef_.round(4),
})
comparison['diff_manual_vs_sklearn'] = (comparison['beta_ridge_manual'] - comparison['beta_ridge_sklearn']).round(6)
comparison

### Ridge при разных $\lambda$: регуляризационный путь

Посмотрим на эффект в динамике — обучим Ridge на сетке $\lambda$ и построим график весов `Latitude`/`Longitude` от $\lambda$.

**В тренажере укажите $\lambda$, давшее зануление для Ridge.**

In [ ]:
lambdas_ridge = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]

ridge_path = []
for lam in lambdas_ridge:
    r = Ridge(alpha=lam)
    r.fit(X_train_scaled, y_train)
    ridge_path.append(r.coef_)

ridge_path = np.array(ridge_path)

plt.figure(figsize=(8, 5))
plt.plot(lambdas_ridge, ridge_path[:, list(labels).index('Latitude')], marker='o', label='Latitude')
plt.plot(lambdas_ridge, ridge_path[:, list(labels).index('Longitude')], marker='o', label='Longitude')
plt.xscale('log')
plt.xlabel('$\lambda$ (лог. шкала)')
plt.ylabel('Вес')
plt.legend()
plt.title('Ridge: регуляризационный путь для Latitude/Longitude');

### Lasso: обнуление при конечном $\lambda$

**Q13.** Обучите `Lasso` на той же сетке `X_train_scaled`, `y_train` для $\lambda \in \{0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0\}$ (`Lasso(alpha=lam, max_iter=20000)`). Для каждого $\lambda$ посчитайте число ненулевых весов и отдельно веса `Latitude`/`Longitude`. 

Какое значение $\lambda$ привело к 0 при Latitude/Longitude хотя бы для одного признака для Lasso?

In [ ]:
from sklearn.linear_model import Lasso

lambdas_lasso = [0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]

lasso_path = []
for lam in lambdas_lasso:
    l = Lasso(alpha=lam, max_iter=20000)
    l.fit(X_train_scaled, y_train)
    lasso_path.append(l.coef_)

lasso_path = np.array(lasso_path)

lasso_report = pd.DataFrame({
    'lambda': lambdas_lasso,
    'n_nonzero': (np.abs(lasso_path) > 1e-8).sum(axis=1),
    'Latitude': lasso_path[:, list(labels).index('Latitude')].round(4),
    'Longitude': lasso_path[:, list(labels).index('Longitude')].round(4),
})
lasso_report

In [ ]:
# Полная картина: что происходит со ВСЕМИ весами, не только Latitude/Longitude
lasso_full = pd.DataFrame(lasso_path, columns=labels)
lasso_full.insert(0, 'lambda', lambdas_lasso)
lasso_full.insert(1, 'n_nonzero', (np.abs(lasso_path) > 1e-8).sum(axis=1))
lasso_full.round(4)

**Возвращаемся к предсказанию из Блока 5.** Посмотрите на полную таблицу выше — картина богаче, чем «регуляризация убирает коррелирующие признаки»:

- **Первым уходит `Population`** (VIF $\approx 1.1$, ни с чем не коррелирует) — обнуляется одним из первых, уже к $\lambda=0.005$, в одиночку.
- **Вторыми, все втроём разом, к $\lambda=0.05$** — `AveRooms`, `AveBedrms` **и** `AveOccup`. Обратите внимание: это смешанная группа — `AveRooms`/`AveBedrms` действительно коррелируют между собой (VIF $\approx 8.1/6.8$), а `AveOccup` не коррелирует ни с чем (VIF $\approx 1.0$) — но Lasso обнуляет их **одновременно**, потому что по итоговому вкладу в качество они сейчас в одной весовой категории, а не потому что как-то связаны друг с другом.
- **`Longitude`** уходит следующим, к $\lambda=0.1$, пока `Latitude` ещё держится на маленьком весе ($-0.0104$).
- **`Latitude` и `HouseAge` обнуляются вместе**, к $\lambda=0.2$ — опять пара, где один признак (`Latitude`) участвовал в сильной корреляции, а другой (`HouseAge`, VIF $\approx 1.24$) — нет, но по вкладу на этом шаге они сравнялись.
- **`MedInc`** держится дольше всех, до $\lambda=1.0$.

Если бы гипотеза «регуляризация убирает именно коррелирующие признаки» была верна, первыми и в чистом виде должны были обнулиться пары `Latitude`/`Longitude` и `AveRooms`/`AveBedrms` — а по факту порядок обнуления перемешивает коррелирующие и некоррелирующие признаки на каждом шаге, и самой первой ушла вообще ни с чем не коррелирующая `Population`.

**Ответ на общий вопрос: нет, регуляризация не обязана убирать только сильно коррелирующие признаки.** Она штрафует веса по их вкладу в снижение ошибки относительно цены $\lambda$ — и обнуляет то, что дешевле всего потерять, а не то, что с чем-то коррелирует. Корреляция — лишь один из способов сделать признак «дешёвым»: если у пары есть партнёр, который покроет большую часть её вклада, индивидуальная потеря от обнуления одного из них меньше. Но точно так же дёшево обнулить и просто слабый, ни с кем не коррелирующий признак — как мы и увидели с `Population`. А сильно коррелирующая, но информативная пара (`Latitude`/`Longitude`) пережила куда более слабые признаки, потому что *вместе* она даёт реальный вклад в качество, даже если по отдельности веса нестабильны (Блок 3, Блок 5).

То есть **корреляция объясняет нестабильность точечной оценки веса** (за это отвечает VIF, Блок 5), но не определяет напрямую порядок обнуления в Lasso — за это отвечает совместный вклад признака (или группы признаков) в качество модели относительно штрафа $\lambda$.

### Бонус: L0 в лоб — точный перебор

L0-регуляризация штрафует не величину веса, а сам факт его использования: $\|\beta\|_0=\sum_j\mathbb{1}[\beta_j\neq 0]$. В общем случае это NP-трудная комбинаторная задача — переборы подмножеств признаков. Но у нас всего 8 признаков, а значит и подмножеств всего $2^8=256$ — переберём их **точно**, а не через приближение.

Для каждого $\lambda$ ищем подмножество признаков $S$, минимизирующее $SSE(S) + \lambda\cdot|S|$, где $SSE(S)$ — сумма квадратов ошибок OLS, обученной только на признаках из $S$ (на `X_train_scaled`, `y_train`).

**Q14 (со звёздочкой).** Реализуйте точный перебор для $\lambda \in \{0,\ 50,\ 200,\ 500,\ 1000,\ 3000,\ 6000,\ 10000,\ 20000\}$ и выведите для каждого $\lambda$ выбранное подмножество признаков.

При каком $\lambda$ множество признаков оказалось пустым (модели дешевле не использовать признаки вовсе)?

In [ ]:
from itertools import combinations

def sse_for_subset(cols, Xtr, ytr):
    y_centered = ytr - ytr.mean()
    if len(cols) == 0:
        return np.sum(y_centered**2)
    Xsub = Xtr[:, cols]
    beta, *_ = np.linalg.lstsq(Xsub, y_centered, rcond=None)
    pred = Xsub @ beta
    return np.sum((y_centered - pred)**2)

feat_idx = list(range(len(labels)))
lambdas_l0 = [0, 50, 200, 500, 1000, 3000, 6000, 10000, 20000]

l0_rows = []
for lam in lambdas_l0:
    best = None
    for k in range(0, len(feat_idx)+1):
        for combo in combinations(feat_idx, k):
            sse = sse_for_subset(list(combo), X_train_scaled, y_train.values)
            obj = sse + lam*k
            if best is None or obj < best[0]:
                best = (obj, combo, sse)
    obj, combo, sse = best
    l0_rows.append({'lambda': lam, 'k': len(combo), 'features': [labels[i] for i in combo], 'SSE': round(sse, 1)})

pd.DataFrame(l0_rows)

На этом всё, друзья! Мы прошли весь путь: от «какой признак самый важный» через «а насколько мы вообще уверены в этой оценке» и «что вообще прогнозирует модель» — до того, что $\beta_0$ и регуляризация делают с этой неуверенностью. 

Вы прошли большой путь и во многом потренировались. Вы молодцы!

Встретимся в домашних заданиях,  \
Ваша команда курса : ) 